# LLM-as-Judge and Faithfulness: RAGAS as a Family of Estimators

The narrative companion to `llm_as_judge_ragas.py`, the canonical reference that **owns every number**
this topic reports. We never redefine the math here — we import the module and walk it section by
section. The evaluation track measured retrieval against **fixed** relevance labels; generation has
none, so we hire an LLM as a **judge** and treat it as a noisy measurement **instrument**. Every RAGAS
metric becomes an *estimator* built from the judge's verdicts, and reading it correctly means
**correcting for the instrument**: debias the verdict (Rogan–Gladen), agree on the protocol
(chance-corrected reliability), calibrate the confidence (the prerequisite's suite), and price the
judge-variance floor (ICC).

The judge here is **synthetic** — a Bernoulli rater of the imported MaxSim oracle whose
sensitivity/specificity we control — which is exactly what lets us prove the estimator theorems
exactly and demonstrate the corrections. A real judge's error rates are unknown and input-dependent;
that gap is the topic's load-bearing honesty flag.

In [ ]:
import pathlib, sys

# Path-robust import: find the module regardless of the kernel's working directory. The module itself
# adds every ancestor notebook dir (the multi-vector subtree + all three eval prereqs) to sys.path.
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "llm-as-judge-ragas",
              pathlib.Path("notebooks/llm-as-judge-ragas")):
    if (_cand / "llm_as_judge_ragas.py").exists():
        sys.path.insert(0, str(_cand.resolve()))
        break

import numpy as np
import llm_as_judge_ragas as J

corpus = J.get_corpus()
print(f"corpus: {corpus['n_queries']} queries x {corpus['n_docs']} docs; "
      f"K={J.K} candidate claims/answer; legs={J.LEG_NAMES}")

## 0. The truth, the candidates, and a perfect judge

The oracle gives a binary truth `Y[q, d] = 1` iff doc `d` is genuinely relevant to query `q`. A
system's top-`K` retrieved docs are the answer's **claims**; the faithfulness *estimand* is the true
supported fraction, which is exactly the imported `precision@k`. A **perfect** judge (sensitivity =
specificity = 1) returns the truth, so its faithfulness reproduces `precision@k` to machine precision
— the collapse anchor.

In [ ]:
for leg in J.LEG_NAMES:
    perfect = J.judged_faithfulness(corpus, leg, J.JUDGE_PERFECT, np.random.default_rng(J.SEED))
    oracle = J.oracle_faithfulness(corpus, leg)
    print(f"{leg:16s} perfect-judge faithfulness == precision@k   (max|diff| = {np.max(np.abs(perfect - oracle)):.1e})")

## 1. The judge as a noisy instrument — Rogan–Gladen debiasing

A real judge has sensitivity `se` and specificity `sp` below 1, so the observed positive rate is
`p_obs = se·π + (1−sp)(1−π)` — a **biased** estimate of the latent prevalence `π`. Inverting gives the
Rogan–Gladen estimate `π̂ = (p_obs + sp − 1)/(se + sp − 1)`. It is unbiased for a homogeneous judge
with known rates, but its variance is inflated by `1/(se+sp−1)²` (the inverse-square Youden index): a
near-useless judge cannot be debiased, only amplified.

In [ ]:
print("Lenient judge (verbosity + position + self-preference bias):")
for leg in J.LEG_NAMES:
    cf = J.corrected_faithfulness(corpus, leg, J.JUDGE_LENIENT, np.random.default_rng(J.SEED))
    print(f"  {leg:16s} naive p_obs={cf['p_obs']:.3f}  ->  Rogan-Gladen pi={cf['pi_corrected']:.3f}"
          f"   (oracle {cf['pi_oracle']:.3f}; Youden J={cf['youden']:.3f}, variance x{cf['var_inflation']:.1f})")

print("\nExact recovery on known (pi, se, sp) — the affine inversion:")
for pi, se, sp in [(0.5, 0.9, 0.9), (0.3, 0.8, 0.85), (0.7, 0.95, 0.6)]:
    pobs = pi * se + (1 - pi) * (1 - sp)
    print(f"  pi={pi}  se={se}  sp={sp}  ->  p_obs={pobs:.3f}  ->  pi_hat={J.rogan_gladen(pobs, se, sp, clip=False):.3f}")

## 2. Agreement is not accuracy — the κ paradox, AC1, and the variance floor

Before trusting a judge we ask whether it *agrees* with another rater. Cohen's
`κ = (p_o − p_e)/(1 − p_e)` chance-corrects raw agreement, but under skewed marginals two raters can
agree on 85% of items yet score κ near zero (Feinstein–Cicchetti). Gwet's AC1 is stable there. A
two-way variance decomposition then exposes the **judge-variance floor**: more queries shrink the
query-sampling term but never the per-item judge noise.

In [ ]:
kp = J.kappa_paradox_tables()
for name in ("balanced", "skewed"):
    t = kp[name]
    print(f"{name:9s}  p_o={t['po']:.2f}  kappa={t['kappa']:.3f}  AC1={t['ac1']:.3f}")
print(f"-> identical agreement (gap {kp['po_gap']:.0e}), but kappa gap {kp['kappa_gap']:.3f} vs AC1 gap {kp['ac1_gap']:.3f}\n")

vc = J.variance_components(J.judge_panel_ratings(corpus, "dense"))
print(f"ICC(2,1) = {vc['icc21']:.3f}   (var_query={vc['var_query']:.4f}  var_judge={vc['var_judge']:.4f}  var_error={vc['var_error']:.4f})")
print("Standard error vs #queries at one judge — the judge floor stays flat:")
for r in J.precision_floor_vs_n(vc, n_judges=1):
    print(f"  Q={r['n']:4d}   SE_query={r['se_query']:.4f}   judge_floor={r['se_floor']:.4f}   SE_total={r['se_total']:.4f}")
bl = J.budget_lever(vc)
print(f"\nBudget lever at {bl['budget']} judge-calls: SE(40q,5j)={bl['se_multi_5j']:.4f} < SE({bl['budget']}q,1j)={bl['se_single_1j']:.4f}? "
      f"{bl['multi_judge_wins']}  (more judges beats more queries for precision)")

## 3. Is the judge's confidence a probability? Calibration and a bias swap test

The judge's stated confidence feeds the prerequisite's calibration suite verbatim. Raw confidence is
over-confident (high ECE) even when it *ranks* well (high AUC) — calibration is orthogonal to ranking,
so a strictly monotone recalibration (Platt) lowers ECE while leaving AUC exactly unchanged. A
**paired swap test** — each claim scored in the first vs the last slot — detects a removable position
bias; a bias-free control does not reject.

In [ ]:
for leg, jp in (("dense", J.JUDGE_LENIENT), ("late_interaction", J.JUDGE_STRICT)):
    t = J.judge_ece_table(corpus, leg, jp)
    print(f"{leg:16s} ECE raw={t['raw']['ece']:.3f} -> platt={t['platt']['ece']:.3f} iso={t['isotonic']['ece']:.3f}"
          f"  |  AUC raw={t['auc_raw']:.4f} == platt={t['auc_platt']:.4f} (ranking preserved)")
print()
for label, leg, jp in (("dense (b_pos>0)", "dense", J.JUDGE_LENIENT),
                       ("dense (b_pos=0 control)", "dense", J.JUDGE_HOMOG)):
    sb = J.swap_test_bias(corpus, leg, jp)
    print(f"swap test {label:24s}: bias={sb['bias']:+.4f}   paired-t p={sb['t_p']:.2e}   permutation p={sb['perm_p']:.2e}")

## 4. Learning the error rates with no gold labels — Dawid–Skene EM

Rogan–Gladen needed the judge's `se`/`sp`. When there is no gold label, the Dawid–Skene latent-class
EM recovers each judge's confusion matrix and the hidden true labels from the **agreement structure
alone** — closing the loop with Section 1. It is identifiable only up to a label permutation (we align
to majority vote) and its likelihood is non-convex.

In [ ]:
pl = J.planted_judge_panel()
ds = J.dawid_skene_em(pl["ratings"])
print("judge |  planted (se, sp)  |  EM-recovered (se, sp)")
for j in range(len(pl["sens"])):
    print(f"  {j}   |  ({pl['sens'][j]:.2f}, {pl['spec'][j]:.2f})   |  ({ds['sens'][j]:.2f}, {ds['spec'][j]:.2f})")
maj = (pl["ratings"].mean(axis=1) >= 0.5).astype(int)
print(f"\nEM label accuracy {np.mean(ds['labels_hard'] == pl['truth']):.3f} >= majority-vote {np.mean(maj == pl['truth']):.3f}"
      f"   (no gold labels used; {ds['n_iter_run']} EM iterations)")

## The headline: correcting for the instrument can flip the ranking

Two systems judged by the *same* lenient judge: the one whose documents are longer/earlier is inflated
more, so the **raw** faithfulness can rank a worse system first. Rogan–Gladen with each system's
audited rates restores the true order. (No natural leg pair flips on this corpus — the judge inflates
all three legs monotonically — so this is a constructed two-system toy, the same honest device the
NDCG topic uses for its convention flips.)

In [ ]:
flip = J.correction_flips_ranking(corpus)
hi = lambda gap: flip["pair"][0] if gap > 0 else flip["pair"][1]
print(f"ranking flip ({flip['kind']}): {flip['pair']}")
print(f"  raw faithfulness gap = {flip['raw_gap']:+.3f}   (favours {hi(flip['raw_gap'])})")
print(f"  corrected gap        = {flip['corr_gap']:+.3f}   (favours {hi(flip['corr_gap'])})")
print(f"  oracle gap           = {flip['oracle_gap']:+.3f}   -> the correction recovers the true order")